# Homework 10

* Please follow the [guidelines](https://fmfi-compbio.github.io/viz/Tutorial1.html\#pokyny-k-zadaniam) for submitting notebooks. 
* Do not forget to switch off AI assistants in Colab `Menu->Tools->Settings->AI Assistance`.
* Use only the libraries imported by our code below.

In [ ]:
# importing libraries
import numpy as np
import pandas as pd
import plotly.express as px

# install the required version of Dash library
! pip install dash==3.2.0

from dash import Dash, dcc, html, Input, Output

## Introduction

In this task, you will create an extended version of the interactive plot from [Lecture 10b](https://colab.research.google.com/github/fmfi-compbio/viz//blob/master/notebooks/L10b_Dash.ipynb) using [Dash](https://dash.plotly.com/) library by Plotly. 
* In the first task you will create a function for creating a table of mathematical functions to be plotted.
* In the second task you will create the interactive plot of these functions.

## Creating the table of functions

Below is function `convert_table` which converts a table of functions from Python dictionary to Pandas DataFrame. We have already used this function in both Lecture 1c a Group tasks G02. You do not need to study its code.

In [ ]:
def convert_table(x, function_dict):
    """ x is a list (or Numpy array) of values of x,
    function_dict is a dictionary containing function names as keys
    and lists of function values as values. The result will be a Pandas 
    DataFrame (table) with each row containing triple x, function, value.
    Zeroes and negative values are masked as missing 
    to avoid problems with logarithmic y axis."""

    # check that all functions have the same number of values as x
    for f in function_dict:
        assert(len(function_dict[f])==len(x))

    # create a wide table with each function as one columns
    functions_wide = pd.DataFrame(function_dict, index=x)
    # reformat to long format 
    #   where each row is a triple x, function name, function value
    functions = (functions_wide.reset_index()
                 .melt(id_vars='index')
                 .rename(columns={'variable':'function', 'index':'x'}))
    # mask values <= 0 as missing values
    val = functions['value']
    functions['value'] = val.mask(val <= 0, np.nan)
    return functions

### Task 1: Implement function `create_functions` [2 points]

Implement function `create_functions` which produces a table similar to table `functions` from [Lecture 10b](https://colab.research.google.com/github/fmfi-compbio/viz//blob/master/notebooks/L10b_Dash.ipynb) based on the following inputs:
* Argument `function_list` is a list of desired function names. Your output table should include exactly these functions. If the value of this argument is `None`, return all functions supported by your code. You should be able to support at least `quadratic` and `cubic` functions used in Lecture 10b, but ideally you should include also other functions from Group tasks G05 or even other functions you like. 
* Arguments `range_end` and `point_num` determine the values of `x` considered. This part of already implemented. We will always use `point_num` values of `x`, where the smallest is `range_end / point_num` and the largest is `point_num`. Argument `point_num` is optional with default value 100. 

The result of the function is a Pandas DataFrame created by function `convert_table` from a Python dictionary. Your task is therefore to create a correct dictionary according to `function_list` and pass it to `convert_table`. You can use parts of the code from Lecture 10b or from Group tasks G05 of your group.


In [ ]:
def create_functions(function_list, range_end, point_num=100):
    # generate values of x:
    x = np.linspace(range_end / point_num, range_end, point_num)

    ### TASK 1 START
    # write your code here
    ### TASK 1 END

### Testing your function

Several cells below test correctness of your function. Each cell calls your function with different arguments and checks the result.

In [ ]:
display("Create and display a small table with 2 functions, each with 3 values")
toy_function_table = create_functions(['quadratic', 'cubic'], 3, point_num=3)
display(toy_function_table)

display("Running various checks on the table")
assert toy_function_table.shape == (6,3)
assert sorted(toy_function_table.columns) == ['function', 'value', 'x']
assert sorted(toy_function_table['function'].unique()) == ['cubic', 'quadratic']
assert sorted(toy_function_table['x'].unique()) == [1, 2, 3]
sel_value = toy_function_table.query('x==2 and function=="quadratic"')['value']
assert sel_value.shape==(1,) and sel_value.iloc[0]==4
display("  OK")

In [ ]:
display("Create and display a small table with 1 function and 2 values")
one_function_table = create_functions(['quadratic'], 100, point_num=2)
display(one_function_table)

display("Running various checks on the table")
assert one_function_table.shape == (2,3)
assert sorted(one_function_table.columns) == ['function', 'value', 'x']
assert sorted(one_function_table['function'].unique()) == ['quadratic']
assert sorted(one_function_table['x'].unique()) == [50, 100]
sel_value = one_function_table.query('x==100 and function=="quadratic"')['value']
assert sel_value.shape==(1,) and sel_value.iloc[0]==10000
display("  OK")

The cell below uses an empty list of function names. Your function should correctly work even in this case and produce a table with three named columns but otherwise empty. Note that `convert_table` produces such an output when given an empty Python dictionary.

In [ ]:
display("Create and display a table with 0 functions")
zero_function_table = create_functions([], 1)
display(zero_function_table)

display("Running various checks on the table")
assert zero_function_table.shape == (0,3)
assert sorted(one_function_table.columns) == ['function', 'value', 'x']
display("  OK")

## Getting the list of all functions

Finally the code below calls your function with argument `None` instead of a list, which should produce a table of all functions your code supports. Names of these functions are stored in `all_functions` list to be used later.

In [ ]:
display("Create a table with all supported functions and extract a list of their names")
all_functions_table = create_functions(None, 100)
all_functions = list(all_functions_table['function'].unique())
print('All functions:', all_functions)
print('Running some checks of all_functions list')
assert 'quadratic' in all_functions
assert 'cubic' in all_functions
print(' OK')

## An example of a plot in Plotly

* Below we use your function `create_functions` to create a table of two functions and plot them on using [Plotly](https://plotly.com/python/plotly-express/).
* We set both axes to be logarithmic.
* You can use parts of this code in Task 2.

In [ ]:
two_functions = create_functions(['quadratic', 'cubic'], 10)
figure = px.line(two_functions, x="x", y="value", color='function') 
figure.update_xaxes(type="log")
figure.update_yaxes(type="log")
figure.update_layout(template="simple_white")
figure.show()

## Task 2: An interactive plot in Plotly Dash [5 points]

* Modify the interactive plot in [Dash](https://dash.plotly.com/) from [Lecture 10b](https://colab.research.google.com/github/fmfi-compbio/viz//blob/master/notebooks/L10b_Dash.ipynb) to plot our functions. 
* Keep `Checklist` element for choosing which functions to plot. It should allow users to select any subset of `all_functions` list created above (remove the code from the lecture for creating `all_functions` list).
* Remove `Input` element for setting the title and do not set the title of the plot.
* Add two new controls for choosing if the x and y axis should be linear or logarithmic. This can be achieved for example by [RadioItems](https://dash.plotly.com/dash-core-components/radioitems) that allow user to choose one from a list of options. Each axis should be controled separately.
* Add a new control for setting the maximum value of x axis. For this you can use [Input](https://dash.plotly.com/dash-core-components/input) component with setting `type='number'`. Beware that the value of this element can be `None`, if no valid number is entered. In such a case use a default value 10. Use setting `min=0.001` to avoid negative values or zero as maximum.
* The `update_figure` function from the lecture uses table `functions` created elsewhere in the notebook and selects some part of it to a new variable `functions_subset`. Instead of this, call your function `create_functions` to directly make a table with the desired functions based on user input. As `range_end`, use the maximum value of x axis chosen by the user. Leave the number of points at the default value.

You can find examples of RadioItems and Input elements in the documentation linked in the text above. 

In [ ]:
### TASK 2 START
# write your code here
### TASK 2 END

## Bonus Task 3: add option for choosing template style 

Create another Dash interactive plot. Do not use any of the interactive features from the previous plot. The plot will contain a fixed set of functions, both axes will be logarithmic and maximum value of x axis will be 10. The plot will have only one interactive element which will allow the user to choose one of the predefined plot styles in Plotly (such as `"plotly"`, `"plotly_white"` etc.) and apply it to the plot. 

* See examples in the [documentation](https://plotly.com/python/templates/) for using these styles (called themes).
* To have two Dash applications in the same notebook, the second needs to use a different port than the default 8050. Therefore the last command should be `app.run_server(mode='inline', port=8051)`.

In [ ]:
### TASK bonus START
# write your code here
### TASK bonus END